In [38]:
pip install -U peft torchao bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [39]:
pip install -U bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
from PIL import Image
from peft import PeftModel
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    BitsAndBytesConfig,
)



BASE = "QCRI/Fanar-2-Oryx-IVU"
LORA_ID = "mabdulaziz499/Warraq-Arabic-HTR-7B"

# 2. Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# 3. Load processor and base model
processor = AutoProcessor.from_pretrained(BASE)
model = AutoModelForImageTextToText.from_pretrained(
    BASE,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

# 4. Attach LoRA adapter
model = PeftModel.from_pretrained(model, LORA_ID)
model.eval()



In [41]:
import json
import torch
from PIL import Image

# 1. Define prompt at module level with valid JSON syntax
EXTRACTION_PROMPT = """
مرفق صورة بطاقة هوية. استخرج البيانات التالية بدقة دون تعديل حتى لو وجدت أخطاء إملائية.
- استخرج فقط بصيغة JSON صالحة دون أي نص إضافي قبل أو بعد الأقواس.
- إذا لم تجد بيانًا معينًا، اترك قيمته نصًا فارغًا "".

المطلوب استخراجه:
{
  "name": "الاسم",
  "DOB": "تاريخ الميلاد",
  "NID": "الرقم القومي",
  "Expiry_date": "تاريخ انتهاء البطاقة"
}
""".strip()

def preprocess_id_image(image_path: str, max_dim: int = 1280) -> Image.Image:
    """Load and clamp image dimensions to control vision token count."""
    image = Image.open(image_path).convert("RGB")
    w, h = image.size
    if max(w, h) > max_dim:
        scale = max_dim / max(w, h)
        image = image.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)
    return image

def extract_from_id(image_path: str) -> str:
    image = preprocess_id_image(image_path)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": EXTRACTION_PROMPT},
            ],
        }
    ]

    # 2. Process and move directly to target device in one call
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")

    # 3. Inference with safety margins
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=processor.tokenizer.pad_token_id or model.config.pad_token_id,
        )

    # 4. Slice prompt tokens from output
    input_len = inputs["input_ids"].shape[1]
    output_text = processor.decode(out[0][input_len:], skip_special_tokens=True).strip()
    
    return output_text

In [ ]:
output = extract_from_id('./ar-ocr-test/WhatsApp Image 2026-10-15 at 11.58.49 PM.jpeg')
print(output)

{
  "name": "AHMAD ABDULRAHMAN AHMAD",
  "DOB": "06/08/1969",
  "NID": "1234567890",
  "Expiry_date": "01/08/2023"
}


In [ ]:
output = extract_from_id('./ar-ocr-test/WhatsApp Image 2026-10-16 at 12.02.58 AM.jpeg')
print(output)

```json
{
  "name": "طارق بن حمد الله بن ياسين الحكيم",
  "DOB": "1397/02/11",
  "NID": "1046403927",
  "Expiry_date": "1443/10/27"
}
```


In [ ]:
output = extract_from_id('./ar-ocr-test/WhatsApp Image 2026-10-16 at 12.00.48 AM.jpeg')
print(output)

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


```json
{
  "name": "عبدالله بن عبد العزيز الشهري",
  "DOB": "",
  "NID": "١٠٠٤٩٥٤٣٢",
  "Expiry_date": ""
}
```


In [ ]:
output = extract_from_id('./ar-ocr-test/WhatsApp Image 2026-10-16 at 12.02.58 AM.jpeg')
print(output)

```json
{
  "name": "طارق بن حمد الله بن ياسين الحكيم",
  "DOB": "1397/02/11",
  "NID": "1046403927",
  "Expiry_date": "1443/10/27"
}
```


In [ ]:
output = extract_from_id('./ar-ocr-test/WhatsApp Image 2026-10-16 at 12.03.33 AM.jpeg')
print(output)

```json
{
  "name": "كريستيانو رونالدو دوس سانتوس أفيرو",
  "DOB": "1985/02/05",
  "NID": "٢٥٣٧٩٢٥٧٢",
  "Expiry_date": "١٤٤٤/٠٦/٠٦"
}
```


In [ ]:
output = extract_from_id('./ar-ocr-test/WhatsApp Image 2026-10-16 at 12.07.25 AM.jpeg')
print(output)

```json
{
  "name": "NOAH SAMI AL-MASRI",
  "DOB": "15/10/2023",
  "NID": "",
  "Expiry_date": ""
}
```
